# jupytertesting.ipynb

Exploratory notebook for inspecting mmWave radar recordings and early processing tests.

> This notebook is kept as an exploration notebook. The maintained acquisition, processing, and plotting code is located in the regular Python scripts in the `Software/` folder.

Copyright (C) 2026 Benjamin Löliger

Author: Benjamin Löliger (bloeliger@ethz.ch)

SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License"); you may
not use this file except in compliance with the License.
You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS, WITHOUT
WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

## Setup


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt, find_peaks

from scipy.interpolate import interp1d



SOFTWARE_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..")) 

## Have a look at the data from the BioGAP!

Loads one BioGAP radar recording and checks the array shape, data type, ADC range, and one raw chirp.

In [ ]:
path_biogap_folder = os.path.join(SOFTWARE_ROOT, "data", "biogap_measurments", "radar_fps_sweep", "radar_invivo_fps_sweep_2026-05-22_18-28")
path_biogap = os.path.join(path_biogap_folder, "fps_200_order_05_rep_1_data.npy")
data_biogap = np.load(path_biogap)

print("\nAs expected the shape is (fps*time, #of AX, #of Chirps, #of Samples):")
print(np.shape(data_biogap))

print("\nSince the raw data is 12 Bits we save it as:")
print(data_biogap.dtype)

print("\nIdealy to not have clipping the values should be 0 < x < 4094")
print(data_biogap.min(), data_biogap.max())
print(np.percentile(data_biogap, [0, 1, 50, 99, 100]))

print("\nThis is what the first 10 chirps look like:")
for chirp in range(10):
    print(chirp, data_biogap[0, 0, chirp, :].astype(int))


print("\nAnd some visualization as well:")

plt.hist(data_biogap.flatten(), bins=200)
plt.title("Raw ADC value histogram")
plt.xlabel("ADC value")
plt.ylabel("Count")
plt.grid(True)
plt.show()

plt.plot(data_biogap[0, 0, 0, :], marker="o")
plt.title("One raw chirp")
plt.xlabel("Sample index")
plt.ylabel("ADC value")
plt.ylim((0,4094))
plt.grid(True)
plt.show()

## And from the DevKit!

Loads one DevKit radar recording and converts the normalized data back to an approximate 12-bit ADC range for comparison with the BioGAP recordings.

In [ ]:
path_devkit_folder = os.path.join(SOFTWARE_ROOT, "data", "dev_kit_measurments", "radar_session_2026-03-06_16-57_n")
path_devkit = os.path.join(path_devkit_folder , "01_200fps_35chrep_128ch_32sa.npy")

data = np.load(path_devkit)

data += 1
data /= 2
data *= 4094
data_devkit = data.astype(np.uint16)

print("\nAs expected the shape is (fps*time, #of AX, #of Chirps, #of Samples):")
print(np.shape(data_devkit))

print("\nSince the raw data is 12 Bits we save it as:")
print(data_devkit.dtype)

print("\nIdealy to not have clipping the values should be 0 < x < 4094")
print(data_devkit.min(), data_devkit.max())
print(np.percentile(data_devkit, [0, 1, 50, 99, 100]))

print("\nThis is what the first 5 chirps look like:")
for chirp in range(5):
    print(chirp, data_devkit[0, 0, chirp, :].astype(int))


print("\nAnd some visualization as well:")

plt.hist(data_devkit.flatten(), bins=200)
plt.title("Raw ADC value histogram")
plt.xlabel("ADC value")
plt.ylabel("Count")
plt.grid(True)
plt.show()

plt.plot(data_devkit[0, 0, 0, :], marker="o")
plt.title("One raw chirp")
plt.xlabel("Sample index")
plt.ylabel("ADC value")
plt.ylim((0,4094))
plt.grid(True)
plt.show()

In [ ]:
CACHE_PATH = os.path.join(
    SOFTWARE_ROOT,
    "data",
    "processed",
    "wrist",
    "preprocessed_maneuver_series.npz"
)

data = np.load(CACHE_PATH, allow_pickle=False)


def load_result(prefix, data):
    avg = data[f"{prefix}_average_pulse"]
    std = data[f"{prefix}_std_pulse"]

    return {
        "name": str(data[f"{prefix}_name"].item()),
        "t": data[f"{prefix}_t"],
        "fs": float(data[f"{prefix}_fs"]),
        "signal": data[f"{prefix}_signal"],
        "systolic_peaks": data[f"{prefix}_systolic_peaks"],
        "diastolic_valleys": data[f"{prefix}_diastolic_valleys"],
        "ibi_time": data[f"{prefix}_ibi_time"],
        "ibi_ms": data[f"{prefix}_ibi_ms"],
        "hr_bpm": data[f"{prefix}_hr_bpm"],
        "beats": data[f"{prefix}_beats"],
        "average_pulse": avg if avg.size > 0 else None,
        "std_pulse": std if std.size > 0 else None,
    }


results = {
    "finger": load_result("finger", data),
    "mmwave": load_result("mmwave", data),
}

## The Waveforms are looking good!

Comparing the data captured by both devkit and biogap, the configurations of the recordings were not the same, this is just to show that both can get data that makes sense!


In [ ]:
lowcut = 0.5
highcut = 6.0
target_bin_index = 1 #THE TARGET BIN HERE IS SET FOR SIMPLICITY
time = 10

data1 = data_devkit
data2 = data_biogap

num_frames, num_ant, num_chirps1, num_samples1 = data1.shape
fps = num_frames/time 

num_frames, num_ant, num_chirps2, num_samples2 = data2.shape


mean_removed1 = data1 - np.mean(data1, axis=-1, keepdims=True)
window = np.hanning(num_samples1)
windowed_data1 = mean_removed1 * window
range_fft1 = np.fft.rfft(windowed_data1, axis=-1)


mean_removed2 = data2 - np.mean(data2, axis=-1, keepdims=True)
window = np.hanning(num_samples2)
windowed_data2 = mean_removed2 * window
range_fft2 = np.fft.rfft(windowed_data2, axis=-1)

mean_energy = np.mean(np.abs(range_fft1[:, 0, 0, 1:6]), axis=0)
target_bin_index = np.argmax(mean_energy) + 1

# Phase extrahieren (erste Antenne, Mittelwert über Chirps)
complex_signal1 = np.mean(range_fft1[:, 0, :, target_bin_index], axis=1)
complex_signal2 = np.mean(range_fft2[:, 0, :, target_bin_index], axis=1)

# 2. Jetzt erst den Winkel ziehen und entrollen
unwrapped_phase1 = np.unwrap(np.angle(complex_signal1))
unwrapped_phase2 = np.unwrap(np.angle(complex_signal2))

RADAR_FREQ_GHZ = 60.75  # e.g. 60 or 77
c = 3e8                                     # speed of light m/s
wavelength_m = c / (RADAR_FREQ_GHZ * 1e9)   # in meters
wavelength_mm = wavelength_m * 1000         # in mm

# Convert unwrapped phase (radians) → displacement (mm)
displacement_mm1 = (wavelength_mm * unwrapped_phase1) / (4 * np.pi)
displacement_mm2 = (wavelength_mm * unwrapped_phase2) / (4 * np.pi)
        

# Filter
nyquist = 0.5 * fps
b, a = butter(N=3, Wn=[lowcut/nyquist, highcut/nyquist], btype='band')
filtered_phase1 = filtfilt(b, a, displacement_mm1)
filtered_phase2 = filtfilt(b, a, displacement_mm2)


# Signal für Bin 1 (oder target_bin_index)
vital_sign_signal1 = filtered_phase1
vital_sign_signal2 = filtered_phase2

unfilterd = unwrapped_phase1

# Polarität anpassen
if np.mean(vital_sign_signal1**3) < 0:
    vital_sign_signal1 *= -1


if np.mean(vital_sign_signal2**3) < 0:
    vital_sign_signal2 *= -1

# Plotting direkt im Notebook
time_axis = np.arange(num_frames) / fps
    
plt.figure(figsize=(14, 4))
plt.plot(time_axis, vital_sign_signal1, color="#0c87df", linewidth=1.5, label='data_devkit')
plt.plot(time_axis, vital_sign_signal2, color="#b41f1f", linewidth=1.5, label='data_biogap')
    

# Styling
plt.title(f"Wave forms captured using the DevKit and BioGap", fontsize=12, fontweight='bold')
plt.xlabel("Time [sec]")
plt.ylabel("Displacement [mm]")
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right')
    
plt.show()

And now comparing the magnitude signal against the Phase signal captured by the biogap sensor

In [ ]:
# ============================================================
# CONFIG
# ============================================================

files = [
    #"radar_invivo_fps_sweep_2026-05-22_17-28/fps_200_order_04_rep_3_data.npy",
    #"radar_invivo_fps_sweep_2026-05-22_17-51/fps_200_order_04_rep_3_data.npy",
    #"radar_session_2026-03-06_17-39_seb/01_200fps_35chrep_128ch_32sa.npy",
    #"radar_session_2026-03-06_17-39_seb/09_200fps_35chrep_128ch_32sa.npy",
    #"radar_session_2026-03-06_17-39_seb/18_200fps_35chrep_128ch_32sa.npy",
    #"radar_session_2026-03-06_16-57_nima/01_200fps_35chrep_128ch_32sa.npy",
    #"radar_session_2026-03-06_16-57_nima/09_200fps_35chrep_128ch_32sa.npy",
    #"radar_session_2026-03-06_16-57_nima/18_200fps_35chrep_128ch_32sa.npy",
    #"radar_session_2026-03-06_17-15_ben/01_200fps_35chrep_128ch_32sa.npy",
    #"radar_session_2026-03-06_17-15_ben/09_200fps_35chrep_128ch_32sa.npy",
    #"radar_session_2026-03-06_17-15_ben/18_200fps_35chrep_128ch_32sa.npy",
    #path_devkit,
    path_biogap,
]

RECORDING_TIME_S = 10

LOWCUT_HZ = 0.5
HIGHCUT_HZ = 8.0
FILTER_ORDER = 4

BIN_SEARCH_START = 1
BIN_SEARCH_END = 4

POINTS_PER_BEAT = 200
MIN_HEART_PERIOD_S = 0.35
PROMINENCE_FACTOR = 0.20


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def bandpass_filter(x, fs, lowcut=0.5, highcut=8.0, order=4, axis=0):
    nyquist = 0.5 * fs

    if highcut >= nyquist:
        highcut = 0.95 * nyquist

    b, a = butter(
        order,
        [lowcut / nyquist, highcut / nyquist],
        btype="band"
    )

    return filtfilt(b, a, x, axis=axis)


def detect_systolic_and_diastolic(signal, fs):
    min_distance = max(1, int(MIN_HEART_PERIOD_S * fs))
    prominence = max(1e-12, PROMINENCE_FACTOR * np.nanstd(signal))

    systolic_peaks, _ = find_peaks(
        signal,
        distance=min_distance,
        prominence=prominence
    )

    diastolic_valleys, _ = find_peaks(
        -signal,
        distance=min_distance,
        prominence=prominence
    )

    return systolic_peaks, diastolic_valleys

def detect_peaks_and_footpoints(signal, fs):
    """
    Detect systolic peaks and one foot/diastolic valley before each peak.
    Better for pulse waveform segmentation.
    """

    min_peak_distance = max(1, int(0.45 * fs))
    prominence = max(1e-12, 0.25 * np.nanstd(signal))

    systolic_peaks, _ = find_peaks(
        signal,
        distance=min_peak_distance,
        prominence=prominence
    )

    if len(systolic_peaks) < 2:
        return systolic_peaks, np.array([], dtype=int)

    footpoints = []

    for i in range(1, len(systolic_peaks)):
        prev_peak = systolic_peaks[i - 1]
        curr_peak = systolic_peaks[i]

        rr = curr_peak - prev_peak

        # Search before current systolic peak
        search_start = curr_peak - int(0.65 * rr)
        search_end = curr_peak - int(0.08 * rr)

        search_start = max(0, search_start)
        search_end = max(search_start + 1, search_end)

        segment = signal[search_start:search_end]

        if len(segment) == 0:
            continue

        foot_idx = search_start + np.nanargmin(segment)
        footpoints.append(foot_idx)

    return systolic_peaks, np.array(footpoints, dtype=int)

def correct_polarity(signal, fs):
    min_distance = max(1, int(MIN_HEART_PERIOD_S * fs))
    prominence = max(1e-12, PROMINENCE_FACTOR * np.nanstd(signal))

    pos_peaks, _ = find_peaks(
        signal,
        distance=min_distance,
        prominence=prominence
    )

    neg_peaks, _ = find_peaks(
        -signal,
        distance=min_distance,
        prominence=prominence
    )

    if len(pos_peaks) == 0 or len(neg_peaks) == 0:
        if np.nanmean(signal ** 3) < 0:
            return -signal
        return signal

    pos_amp = np.nanmean(signal[pos_peaks])
    neg_amp = np.nanmean(-signal[neg_peaks])

    if neg_amp > pos_amp:
        signal = -signal

    return signal


def extract_normalized_beats(signal, valley_indices, points_per_beat=200):
    beats = []

    valley_indices = np.sort(valley_indices)

    for i in range(len(valley_indices) - 1):
        start = valley_indices[i]
        end = valley_indices[i + 1]

        beat = signal[start:end]

        if len(beat) < 10:
            continue

        beat = beat - np.nanmin(beat)
        beat = beat / (np.nanmax(beat) + 1e-12)

        x_old = np.linspace(0, 1, len(beat))
        x_new = np.linspace(0, 1, points_per_beat)

        beat_interp = interp1d(x_old, beat, kind="linear")(x_new)
        beats.append(beat_interp)

    return np.array(beats)


def process_mmwave_magnitude_file(
    file_path,
    recording_time_s,
    bin_search_start=1,
    bin_search_end=None,
):
    """
    Magnitude-based mmWave pulse waveform processing.

    Expected raw data shape:
    frames x antennas x chirps x samples
    """

    data = np.load(file_path)

    num_frames, num_antennas, num_chirps, num_samples = data.shape
    fps = num_frames / recording_time_s
    time_axis = np.arange(num_frames) / fps

    print("\nProcessing magnitude:", file_path)
    print("Raw shape:", data.shape)
    print("Estimated fps:", fps)

    # ------------------------------------------------------------
    # 1. DC removal across samples per chirp
    # ------------------------------------------------------------
    mean_removed = data - np.mean(data, axis=-1, keepdims=True)

    # ------------------------------------------------------------
    # 2. Windowing across ADC samples
    # ------------------------------------------------------------
    window = np.hanning(num_samples)
    windowed_data = mean_removed * window

    # ------------------------------------------------------------
    # 3. Range FFT along sample dimension
    # Shape: frames x antennas x chirps x range_bins
    # ------------------------------------------------------------
    range_fft = np.fft.rfft(windowed_data, axis=-1)

    # ------------------------------------------------------------
    # 4. Magnitude extraction and averaging over chirps
    # Shape: frames x antennas x range_bins
    # ------------------------------------------------------------
    magnitude = np.abs(range_fft)
    magnitude_mean = np.mean(magnitude, axis=2)

    # ------------------------------------------------------------
    # 5. Bandpass filtering, 4th-order Butterworth 0.5–8 Hz
    # ------------------------------------------------------------
    filtered_magnitude = bandpass_filter(
        magnitude_mean,
        fps,
        LOWCUT_HZ,
        HIGHCUT_HZ,
        order=FILTER_ORDER,
        axis=0
    )

    # ------------------------------------------------------------
    # 6. Automatic antenna/range-bin selection
    # Use highest peak-to-peak amplitude of filtered magnitude signal
    # ------------------------------------------------------------
    num_bins = filtered_magnitude.shape[2]

    if bin_search_end is None:
        bin_search_end = num_bins

    bin_search_end = min(bin_search_end, num_bins)

    search_data = filtered_magnitude[:, :, bin_search_start:bin_search_end]

    peak_to_peak = np.nanmax(search_data, axis=0) - np.nanmin(search_data, axis=0)

    best_ant_rel, best_bin_rel = np.unravel_index(
        np.nanargmax(peak_to_peak),
        peak_to_peak.shape
    )

    best_ant = best_ant_rel
    best_bin = best_bin_rel + bin_search_start

    vital_signal_mag = filtered_magnitude[:, best_ant, best_bin]

    print("Selected antenna:", best_ant)
    print("Selected range bin:", best_bin)
    print("Peak-to-peak magnitude:", peak_to_peak[best_ant_rel, best_bin_rel], "a.u.")

    # ------------------------------------------------------------
    # 7. Polarity correction
    # Magnitude can still appear inverted after filtering.
    # ------------------------------------------------------------
    vital_signal_mag = correct_polarity(vital_signal_mag, fps)

    # ------------------------------------------------------------
    # 8. Peak detection
    # ------------------------------------------------------------
    systolic_peaks, diastolic_valleys = detect_peaks_and_footpoints(
        vital_signal_mag,
        fps
    )

    print("Detected systolic peaks:", len(systolic_peaks))
    print("Detected diastolic valleys:", len(diastolic_valleys))

    # ------------------------------------------------------------
    # 9. IBI / HR from diastolic valleys
    # ------------------------------------------------------------
    if len(diastolic_valleys) > 1:
        ibi_s = np.diff(time_axis[diastolic_valleys])
        hr_bpm = 60.0 / ibi_s
    else:
        ibi_s = np.array([])
        hr_bpm = np.array([])

    # ------------------------------------------------------------
    # 10. Beat morphology extraction
    # ------------------------------------------------------------
    beats = extract_normalized_beats(
        vital_signal_mag,
        diastolic_valleys,
        points_per_beat=POINTS_PER_BEAT
    )

    if len(beats) > 0:
        average_pulse = np.nanmean(beats, axis=0)
        std_pulse = np.nanstd(beats, axis=0)
    else:
        average_pulse = None
        std_pulse = None

    return {
        "file_path": file_path,
        "data_shape": data.shape,
        "fps": fps,
        "time_axis": time_axis,
        "filtered_all_bins": filtered_magnitude,
        "vital_signal": vital_signal_mag,
        "best_ant": best_ant,
        "best_bin": best_bin,
        "systolic_peaks": systolic_peaks,
        "diastolic_valleys": diastolic_valleys,
        "ibi_s": ibi_s,
        "hr_bpm": hr_bpm,
        "beats": beats,
        "average_pulse": average_pulse,
        "std_pulse": std_pulse,
    }


# ============================================================
# MAIN LOOP
# ============================================================

results_mag = []

for file_path in files:
    result = process_mmwave_magnitude_file(
        file_path=file_path,
        recording_time_s=RECORDING_TIME_S,
        bin_search_start=BIN_SEARCH_START,
        bin_search_end=BIN_SEARCH_END,
    )

    results_mag.append(result)

    t = result["time_axis"]
    signal = result["vital_signal"]
    systolic_peaks = result["systolic_peaks"]
    diastolic_valleys = result["diastolic_valleys"]

    # ------------------------------------------------------------
    # Plot processed magnitude pulse signal with detected peaks
    # ------------------------------------------------------------
    plt.figure(figsize=(12, 4))

    plt.plot(
        t,
        signal,
        linewidth=1.5,
        label=f"Magnitude signal, ant {result['best_ant']}, bin {result['best_bin']}"
    )

    plt.plot(
        t[systolic_peaks],
        signal[systolic_peaks],
        "x",
        label="Systolic peaks"
    )

    plt.plot(
        t[diastolic_valleys],
        signal[diastolic_valleys],
        "o",
        markersize=4,
        label="Diastolic valleys"
    )

    plt.title("Processed mmWave magnitude pulse signal")
    plt.xlabel("Time [s]")
    plt.ylabel("Magnitude [a.u.]")
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ------------------------------------------------------------
    # Plot average magnitude pulse morphology
    # ------------------------------------------------------------
    if result["average_pulse"] is not None:
        x_norm = np.linspace(0, 1, POINTS_PER_BEAT)

        plt.figure(figsize=(7, 4))

        plt.plot(
            x_norm,
            result["average_pulse"],
            linewidth=2,
            label="Average normalized magnitude pulse"
        )

        plt.fill_between(
            x_norm,
            result["average_pulse"] - result["std_pulse"],
            result["average_pulse"] + result["std_pulse"],
            alpha=0.25,
            label="±1 std"
        )

        plt.title("Average normalized magnitude waveform")
        plt.xlabel("Normalized beat time")
        plt.ylabel("Normalized amplitude")
        plt.grid(True, linestyle=":", alpha=0.6)
        plt.legend()
        plt.tight_layout()
        plt.show()

    # ------------------------------------------------------------
    # Optional HR plot
    # ------------------------------------------------------------
    if len(result["hr_bpm"]) > 0:
        ibi_time = t[diastolic_valleys[1:]]

        plt.figure(figsize=(10, 3))

        plt.plot(
            ibi_time,
            result["hr_bpm"],
            marker="o",
            linewidth=1.2
        )

        plt.title("Heart rate estimated from magnitude valleys")
        plt.xlabel("Time [s]")
        plt.ylabel("Heart rate [bpm]")
        plt.grid(True, linestyle=":", alpha=0.6)
        plt.tight_layout()
        plt.show()

In [ ]:
# ============================================================
# CONFIG
# ============================================================

files = [
    #"radar_invivo_fps_sweep_2026-05-22_17-28/fps_200_order_04_rep_3_data.npy",
    #"radar_invivo_fps_sweep_2026-05-22_17-51/fps_200_order_04_rep_3_data.npy",
    #"radar_session_2026-03-06_17-39_seb/01_200fps_35chrep_128ch_32sa.npy",
    #"radar_session_2026-03-06_17-39_seb/09_200fps_35chrep_128ch_32sa.npy",
    #"radar_session_2026-03-06_17-39_seb/18_200fps_35chrep_128ch_32sa.npy",
    #"radar_session_2026-03-06_16-57_nima/01_200fps_35chrep_128ch_32sa.npy",
    #"radar_session_2026-03-06_16-57_nima/09_200fps_35chrep_128ch_32sa.npy",
    #"radar_session_2026-03-06_16-57_nima/18_200fps_35chrep_128ch_32sa.npy",
    #"radar_session_2026-03-06_17-15_ben/01_200fps_35chrep_128ch_32sa.npy",
    #"radar_session_2026-03-06_17-15_ben/09_200fps_35chrep_128ch_32sa.npy",
    #"radar_session_2026-03-06_17-15_ben/18_200fps_35chrep_128ch_32sa.npy",
    #path_devkit,
    path_biogap,
]

RECORDING_TIME_S = 10

RADAR_FREQ_GHZ = 60.75

LOWCUT_HZ = 0.5
HIGHCUT_HZ = 8.0
FILTER_ORDER = 4

# Bin selection
BIN_SEARCH_START = 1      # skip DC / very near bin
BIN_SEARCH_END = 3     # None = all available bins

# Beat segmentation
POINTS_PER_BEAT = 200

# Peak detection
MIN_HEART_PERIOD_S = 0.35   # 0.35 s = ~171 bpm max
PROMINENCE_FACTOR = 0.20


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def phase_to_displacement_mm(unwrapped_phase, radar_freq_ghz):
    c = 3e8
    wavelength_m = c / (radar_freq_ghz * 1e9)
    wavelength_mm = wavelength_m * 1000.0

    displacement_mm = (wavelength_mm * unwrapped_phase) / (4 * np.pi)

    return displacement_mm


def process_mmwave_file(
    file_path,
    recording_time_s,
    radar_freq_ghz=60.75,
    bin_search_start=1,
    bin_search_end=None,
):
    """
    Paper-like mmWave processing pipeline.

    Expected raw data shape:
    frames x antennas x chirps x samples
    """

    data = np.load(file_path)

    num_frames, num_antennas, num_chirps, num_samples = data.shape
    fps = num_frames / recording_time_s
    time_axis = np.arange(num_frames) / fps

    print("\nProcessing:", file_path)
    print("Raw shape:", data.shape)
    print("Estimated fps:", fps)

    # ------------------------------------------------------------
    # 1. DC removal across samples per chirp
    # ------------------------------------------------------------
    mean_removed = data - np.mean(data, axis=-1, keepdims=True)

    # ------------------------------------------------------------
    # 2. Windowing across ADC samples
    # ------------------------------------------------------------
    window = np.hanning(num_samples)
    windowed_data = mean_removed * window

    # ------------------------------------------------------------
    # 3. Range FFT along sample dimension
    # Shape: frames x antennas x chirps x range_bins
    # ------------------------------------------------------------
    range_fft = np.fft.rfft(windowed_data, axis=-1)

    # ------------------------------------------------------------
    # 4. Complex averaging over chirps
    # Shape: frames x antennas x range_bins
    # ------------------------------------------------------------
    complex_mean = np.mean(range_fft, axis=2)

    # ------------------------------------------------------------
    # 5. Phase extraction and temporal unwrapping
    # ------------------------------------------------------------
    phase = np.angle(complex_mean)
    unwrapped_phase = np.unwrap(phase, axis=0)

    # ------------------------------------------------------------
    # 6. Convert phase to displacement
    # Shape: frames x antennas x range_bins
    # ------------------------------------------------------------
    displacement_mm = phase_to_displacement_mm(
        unwrapped_phase,
        radar_freq_ghz
    )

    # ------------------------------------------------------------
    # 7. Bandpass filtering, 4th-order Butterworth 0.5–8 Hz
    # ------------------------------------------------------------
    filtered_phase = bandpass_filter(
        displacement_mm,
        fps,
        LOWCUT_HZ,
        HIGHCUT_HZ,
        order=FILTER_ORDER,
        axis=0
    )

    # ------------------------------------------------------------
    # 8. Automatic antenna/range-bin selection
    # Paper-like: highest peak-to-peak amplitude of pulsatile signal
    # ------------------------------------------------------------
    num_bins = filtered_phase.shape[2]

    if bin_search_end is None:
        bin_search_end = num_bins

    bin_search_end = min(bin_search_end, num_bins)

    search_data = filtered_phase[:, :, bin_search_start:bin_search_end]

    peak_to_peak = np.nanmax(search_data, axis=0) - np.nanmin(search_data, axis=0)

    best_ant_rel, best_bin_rel = np.unravel_index(
        np.nanargmax(peak_to_peak),
        peak_to_peak.shape
    )

    best_ant = best_ant_rel
    best_bin = best_bin_rel + bin_search_start

    vital_signal = filtered_phase[:, best_ant, best_bin]

    print("Selected antenna:", best_ant)
    print("Selected range bin:", best_bin)
    print("Peak-to-peak amplitude:", peak_to_peak[best_ant_rel, best_bin_rel], "mm")

    # ------------------------------------------------------------
    # 9. Polarity correction
    # ------------------------------------------------------------
    vital_signal = correct_polarity(vital_signal, fps)

    # ------------------------------------------------------------
    # 10. Peak detection
    # ------------------------------------------------------------
    systolic_peaks, diastolic_valleys = detect_peaks_and_footpoints(
        vital_signal,
        fps
    )

    print("Detected systolic peaks:", len(systolic_peaks))
    print("Detected diastolic valleys:", len(diastolic_valleys))

    # ------------------------------------------------------------
    # 11. IBI from diastolic valleys
    # ------------------------------------------------------------
    if len(diastolic_valleys) > 1:
        ibi_s = np.diff(time_axis[diastolic_valleys])
        hr_bpm = 60.0 / ibi_s
    else:
        ibi_s = np.array([])
        hr_bpm = np.array([])

    # ------------------------------------------------------------
    # 12. Beat morphology extraction
    # ------------------------------------------------------------
    beats = extract_normalized_beats(
        vital_signal,
        diastolic_valleys,
        points_per_beat=POINTS_PER_BEAT
    )

    if len(beats) > 0:
        average_pulse = np.nanmean(beats, axis=0)
        std_pulse = np.nanstd(beats, axis=0)
    else:
        average_pulse = None
        std_pulse = None

    return {
        "file_path": file_path,
        "data_shape": data.shape,
        "fps": fps,
        "time_axis": time_axis,
        "filtered_all_bins": filtered_phase,
        "vital_signal": vital_signal,
        "best_ant": best_ant,
        "best_bin": best_bin,
        "systolic_peaks": systolic_peaks,
        "diastolic_valleys": diastolic_valleys,
        "ibi_s": ibi_s,
        "hr_bpm": hr_bpm,
        "beats": beats,
        "average_pulse": average_pulse,
        "std_pulse": std_pulse,
    }


# ============================================================
# MAIN LOOP
# ============================================================

results = []

for file_path in files:
    result = process_mmwave_file(
        file_path=file_path,
        recording_time_s=RECORDING_TIME_S,
        radar_freq_ghz=RADAR_FREQ_GHZ,
        bin_search_start=BIN_SEARCH_START,
        bin_search_end=BIN_SEARCH_END,
    )

    results.append(result)

    t = result["time_axis"]
    signal = result["vital_signal"]
    systolic_peaks = result["systolic_peaks"]
    diastolic_valleys = result["diastolic_valleys"]

    # ------------------------------------------------------------
    # Plot processed pulse signal with detected peaks
    # ------------------------------------------------------------
    plt.figure(figsize=(12, 4))

    plt.plot(
        t,
        signal,
        linewidth=1.5,
        label=f"mmWave phase signal, ant {result['best_ant']}, bin {result['best_bin']}"
    )

    plt.plot(
        t[systolic_peaks],
        signal[systolic_peaks],
        "x",
        label="Systolic peaks"
    )

    plt.plot(
        t[diastolic_valleys],
        signal[diastolic_valleys],
        "o",
        markersize=4,
        label="Diastolic valleys"
    )

    plt.title("Processed mmWave pulse signal")
    plt.xlabel("Time [s]")
    plt.xlim((0,10))
    plt.ylabel("Displacement [mm]")
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    #plt.show()

    # ------------------------------------------------------------
    # Plot average pulse morphology
    # ------------------------------------------------------------
    if result["average_pulse"] is not None:
        x_norm = np.linspace(0, 1, POINTS_PER_BEAT)

        plt.figure(figsize=(7, 4))

        plt.plot(
            x_norm,
            result["average_pulse"],
            linewidth=2,
            label="Average normalized pulse"
        )

        plt.fill_between(
            x_norm,
            result["average_pulse"] - result["std_pulse"],
            result["average_pulse"] + result["std_pulse"],
            alpha=0.25,
            label="±1 std"
        )

        plt.title("Average normalized pulse waveform")
        plt.xlabel("Normalized beat time")
        plt.ylabel("Normalized amplitude")
        plt.grid(True, linestyle=":", alpha=0.6)
        plt.legend()
        plt.tight_layout()
        #plt.show()

    # ------------------------------------------------------------
    # Optional HR / IBI plot
    # ------------------------------------------------------------
    if len(result["hr_bpm"]) > 0:
        ibi_time = t[diastolic_valleys[1:]]

        plt.figure(figsize=(10, 3))

        plt.plot(
            ibi_time,
            result["hr_bpm"],
            marker="o",
            linewidth=1.2
        )

        plt.title("Heart rate estimated from diastolic valleys")
        plt.xlabel("Time [s]")
        plt.ylabel("Heart rate [bpm]")
        plt.grid(True, linestyle=":", alpha=0.6)
        plt.tight_layout()
        #plt.show()

In [ ]:
# Parameter (bleiben für alle gleich zum Vergleich)
lowcut = 0.5
highcut = 8.0
target_bin_index = 1
time = 10


files = [path_biogap]

sync_state = np.load(os.path.join(path_biogap_folder, "fps_200_order_05_rep_1_sync_state.npy"))
plt.figure(figsize=(12,2))
plt.step(np.arange(len(sync_state)), sync_state, where="post")
plt.xlabel("Frame index")
plt.ylabel("Sync state")
plt.grid(True)
plt.show()


#print(f"Verarbeite {len(files)} Dateien direkt im Notebook...")

for filename in files:
    # 1. Laden
    #file_path = os.path.join(input_folder, filename)
    data = np.load(filename)
    #print(data.dtype)
    #print(data.min(), data.max())
    #print(np.percentile(data, [0, 1, 50, 99, 100]))
    #print(np.shape(data))
    #print(data)
    # Dimensionen automatisch erkennen
    # Erwartet (Frames, Antennas, Chirps, Samples)
    num_frames, num_ant, num_chirps, num_samples = data.shape
    fps = num_frames/time 
    
    # 2. Algorithmus (wie in deinem funktionierenden Code)
    mean_removed = data - np.mean(data, axis=-1, keepdims=True)
    window = np.hanning(num_samples)
    windowed_data = mean_removed * window
    range_fft = np.fft.rfft(windowed_data, axis=-1)
    
    # Phase extrahieren (erste Antenne, Mittelwert über Chirps)
    phase_data = np.angle(range_fft[:, 0, :, :]) 
    phase_time_series = np.mean(phase_data, axis=1)
    unwrapped_phase = np.unwrap(phase_time_series, axis=0)

    RADAR_FREQ_GHZ = 60.75  # e.g. 60 or 77
    c = 3e8                                     # speed of light m/s
    wavelength_m = c / (RADAR_FREQ_GHZ * 1e9)  # in meters
    wavelength_mm = wavelength_m * 1000         # in mm

    # Convert unwrapped phase (radians) → displacement (mm)
    displacement_mm = (wavelength_mm * unwrapped_phase) / (4 * np.pi)

    magnitude = np.sqrt(np.square(np.real(range_fft[:, 0, :, :])) + np.square(np.imag(range_fft[:, 0, :, :])))
    magnitude_time_series = np.mean(magnitude, axis=1)


    # Filter
    nyquist = 0.5 * fps
    b, a = butter(N=4, Wn=[lowcut/nyquist, highcut/nyquist], btype='band')
    filtered_phase = filtfilt(b, a, displacement_mm, axis=0)

    filtered_magnitude = filtfilt(b,a ,magnitude_time_series, axis = 0)


    # Signal für Bin 1 (oder target_bin_index)
    vital_sign_signal = filtered_phase[:, target_bin_index]

    vital_sign_mag = filtered_magnitude[:, target_bin_index]

    # Polarität anpassen
    if np.mean(vital_sign_signal**3) < 0:
        vital_sign_signal *= -1


    #if np.mean(vital_sign_mag**3) < 0:
    #    vital_sign_mag *= -1

    # 3. Plotting direkt im Notebook
    time_axis = np.arange(num_frames) / fps

    fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    # Phase-Plot
    axs[0].plot(
        time_axis,
        vital_sign_signal,
        color="#0c87df",
        linewidth=1.5,
        label="Heartbeat-Signal Phase"
    )

    axs[0].set_title("Phase Signal", fontsize=12, fontweight="bold")
    axs[0].set_ylabel("Displacement [mm]")
    axs[0].grid(True, linestyle=":", alpha=0.6)
    #axs[0].legend(loc="upper right")

    # Magnitude-Plot
    axs[1].plot(
        time_axis,
        vital_sign_mag,
        color="#b41f1f",
        linewidth=1.5,
        label="Heartbeat-Signal Magnitude"
    )

    axs[1].set_title("Magnitude Signal", fontsize=12, fontweight="bold")
    axs[1].set_xlabel("Time [sec]")
    axs[1].set_ylabel("Magnitude [a.u.]")
    axs[1].grid(True, linestyle=":", alpha=0.6)
    #axs[1].legend(loc="upper right")

    fig.suptitle("Phase vs Magnitude", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

## Below is Code that was used to explore the devkit data and better understand config parameters


In [ ]:
def extract_average_pulse_waveform(signal, fps, num_points=100):
    # 1. Filtern
    filtered = bandpass_filter(signal, fs=fps)
    
    # 2. Peaks finden (mit Prominence für Stabilität)
    min_peak_distance = max(1, int(0.4 * fps))
    prominence = max(1e-9, 0.2 * np.std(filtered))
    peaks, _ = find_peaks(filtered, distance=min_peak_distance, prominence=prominence)
    
    if len(peaks) < 3: return None

    # 3. Segmente extrahieren und auf 100 Punkte resamplen
    beats = []
    for i in range(len(peaks) - 1):
        segment = filtered[peaks[i]:peaks[i+1]]
        if len(segment) < 4: continue
        
        x_old = np.linspace(0, 1, len(segment))
        x_new = np.linspace(0, 1, num_points)
        segment_resampled = np.interp(x_new, x_old, segment)
        beats.append(segment_resampled)
    
    if len(beats) < 2: return None
    
    beats = np.vstack(beats)
    mean_waveform = np.mean(beats, axis=0)
    
    # 4. Invertierung prüfen (Soll nach oben zeigen)
    k = max(3, int(0.10 * num_points))
    slope = np.polyfit(np.arange(k), mean_waveform[:k], 1)[0]
    if slope < 0:
        beats = -beats
        mean_waveform = np.mean(beats, axis=0)
        
    std_waveform = np.std(beats, axis=0)
    return np.linspace(0, 100, num_points), mean_waveform, std_waveform, len(beats)

In [ ]:
def plot_peak_to_peak(file_path, fps, range_bin=1):
    # Daten laden (shape: frames, antennas, chirps, samples)
    data = np.load(file_path)
    
    # Phase extrahieren (wie in deinem RadarReader)
    # Wir nehmen Antenne 0 und den gewählten Range-Bin
    # FFT über die Samples pro Chirp
    range_fft = np.fft.fft(data[:, 0, :, :], axis=-1)
    bin_values = range_fft[:, :, range_bin] # Alle Chirps des Bins
    
    # Mittelwert über Chirps und Phase berechnen
    #print(bin_values.shape)
   # print(bin_values[:, :32].shape)
    complex_mean = np.mean(bin_values[:, :32], axis=1)
    phase_signal = np.angle(complex_mean)
    
    # WICHTIG: Unwrapping um 2pi-Sprünge zu entfernen
    unwrapped_phase = np.unwrap(phase_signal)

    RADAR_FREQ_GHZ = 60.75  # e.g. 60 or 77

    c = 3e8                                     # speed of light m/s
    wavelength_m = c / (RADAR_FREQ_GHZ * 1e9)  # in meters
    wavelength_mm = wavelength_m * 1000         # in mm

    # Convert unwrapped phase (radians) → displacement (mm)
    displacement_mm = (wavelength_mm * unwrapped_phase) / (4 * np.pi)
        
    # Durchschnittlichen Beat berechnen
    result = extract_average_pulse_waveform(displacement_mm, fps)
    

    if result:
        x_axis, mean_beat, std_beat, num_beats = result

        return x_axis, np.max(mean_beat) - np.min(mean_beat)
        
    else:
        print("Konnte keine sauberen Beats für dieses File finden.")

In [ ]:
inputs = [path_devkit_folder] # add up to two more for comparison
time = 10

x = np.linspace(1,18,18)


fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes_flat = axes.flatten()

for j, input_folder in enumerate(inputs):
    peak_to_peak = []
    ax = axes_flat[j]
    files = sorted([f for f in os.listdir(input_folder) if f.endswith('.npy')])
    for i, filename in enumerate(files):

        file_path = os.path.join(input_folder, filename)
        data = np.load(file_path)
        

        num_frames, num_ant, num_chirps, num_samples = data.shape
        fps = num_frames/time 
        x_axis, ptp = plot_peak_to_peak(file_path=file_path, fps =fps)
        
        peak_to_peak.append(ptp)

    peak_to_peak = np.array(peak_to_peak)
    
    highlight_idx = [0, 8, 17]
    
    ax.plot(x, peak_to_peak,'o', color='blue', lw=2)

    ax.plot(x[highlight_idx], peak_to_peak[highlight_idx], 'o', color='red', 
            markersize=10, zorder=5, label='refrences')

    ax.set_title(f"{input_folder.split('/')[0]}", fontsize=10)
    ax.set_xlabel("Szenario")
    ax.set_ylabel("Peak to Peak (mm)")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)



plt.tight_layout(pad=3.0) # Erhöht den Abstand zwischen den Plots
plt.subplots_adjust(top=0.95) # Platz für eine Hauptüberschrift

plt.show()

In [ ]:
def extract_middle_pulse_waveform(signal, fps, num_points=100):
    # 1. Filtern
    #filtered = bandpass_filter(signal, fps=fps)
    
    # 2. Peaks finden (mit Prominence für Stabilität)
    min_peak_distance = max(1, int(0.4 * fps))
    prominence = max(1e-9, 0.2 * np.std(signal))
    peaks, _ = find_peaks(signal, distance=min_peak_distance, prominence=prominence)
    
    if len(peaks) < 3: return None

    # 3. Segmente extrahieren und auf 100 Punkte resamplen
    beats = []
    for i in range(len(peaks) - 1):
        segment = signal[peaks[i]:peaks[i+1]]
        if len(segment) < 4: continue
        
        x_old = np.linspace(0, 1, len(segment))
        x_new = np.linspace(0, 1, num_points)
        segment_resampled = np.interp(x_new, x_old, segment)
        beats.append(segment_resampled)
    
    if len(beats) < 2: return None
    
    beats = np.vstack(beats)
    mean_waveform = np.mean(beats, axis=0)
    
    # 4. Invertierung prüfen (Soll nach oben zeigen)
    k = max(3, int(0.10 * num_points))
    slope = np.polyfit(np.arange(k), mean_waveform[:k], 1)[0]
    if slope < 0:
        beats = -beats

    correlations = np.array([
        np.mean([np.corrcoef(beats[i], beats[j])[0, 1] 
                 for j in range(len(beats)) if i != j])
        for i in range(len(beats))
    ])
    median_beat = beats[np.argmax(correlations)]

    return np.linspace(0, 100, num_points), median_beat


def plot_chirps(ax, file_path, fps, ref ,range_bin=1):
    # Daten laden (shape: frames, antennas, chirps, samples)
    data = np.load(file_path)
    
    # Phase extrahieren (wie in deinem RadarReader)
    # Wir nehmen Antenne 0 und den gewählten Range-Bin
    # FFT über die Samples pro Chirp
    range_fft = np.fft.fft(data[:, 0, :, :], axis=-1)
    bin_values = range_fft[:, :, range_bin] # Alle Chirps des Bins
    
    # Mittelwert über Chirps und Phase berechnen
    #print(bin_values.shape)
   # print(bin_values[:, :32].shape)
    complex_mean_1 = np.mean(bin_values[:, :1], axis=1)
    complex_mean_32 = np.mean(bin_values[:, :32], axis=1)
    complex_mean_all = np.mean(bin_values, axis=1)
    phase_signal_1 = np.angle(complex_mean_1)
    phase_signal_32 = np.angle(complex_mean_32)
    phase_signal_all = np.angle(complex_mean_all)
    
    # WICHTIG: Unwrapping um 2pi-Sprünge zu entfernen
    unwrapped_phase_1 = np.unwrap(phase_signal_1)
    unwrapped_phase_32 = np.unwrap(phase_signal_32)
    unwrapped_phase_all = np.unwrap(phase_signal_all)

    RADAR_FREQ_GHZ = 60.75  # e.g. 60 or 77

    c = 3e8                                     # speed of light m/s
    wavelength_m = c / (RADAR_FREQ_GHZ * 1e9)  # in meters
    wavelength_mm = wavelength_m * 1000         # in mm

    # Convert unwrapped phase (radians) → displacement (mm)
    displacement_mm_1 = (wavelength_mm * unwrapped_phase_1) / (4 * np.pi)
    displacement_mm_32 = (wavelength_mm * unwrapped_phase_32) / (4 * np.pi)
    displacement_mm_all = (wavelength_mm * unwrapped_phase_all) / (4 * np.pi)
        
    #print(displacement_mm_1.shape, '  ',displacement_mm_32.shape, '  ',displacement_mm_all.shape,)
        
    # Durchschnittlichen Beat berechnen
    #result = extract_average_pulse_waveform(displacement_mm, fps)

    x_axis, wave_1 = extract_middle_pulse_waveform(displacement_mm_1, fps)
    _, wave_32 = extract_middle_pulse_waveform(displacement_mm_32, fps)
    _, wave_all = extract_middle_pulse_waveform(displacement_mm_all, fps)
    wave_1 = (wave_1 - np.min(wave_1)) / (np.max(wave_1) - np.min(wave_1))
    wave_32 = (wave_32 - np.min(wave_32)) / (np.max(wave_32) - np.min(wave_32))
    wave_all = (wave_all - np.min(wave_all)) / (np.max(wave_all) - np.min(wave_all))
    '''
    peak = np.max(np.abs(wave_1))
    if peak > 0:
        wave_1 = wave_1 / peak

    peak = np.max(np.abs(wave_32))
    if peak > 0:
        wave_32 = wave_32 / peak

    peak = np.max(np.abs(wave_all))
    if peak > 0:
        wave_all = wave_all / peak
    '''
    ax.plot(x_axis, wave_1, color='red', lw=2, label='1')
    ax.plot(x_axis, wave_32, color='green', lw=2, label='32')
    ax.plot(x_axis, wave_all, color='blue', lw=2, label='all')
    ax.plot(x_axis, ref, color='black', lw=2, label='ref', linestyle='dashed')



    ax.set_title(f"{file_path.split('\\')[-1]}", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    
    #    return x_axis, np.max(mean_beat) - np.min(mean_beat)


In [ ]:
input_folder = path_devkit_folder

time = 10

files = sorted([f for f in os.listdir(input_folder) if f.endswith('.npy')])

fig, axes = plt.subplots(6, 3, figsize=(15, 22))
axes_flat = axes.flatten()

data = np.load(os.path.join(input_folder, files[0]))
    
# Phase extrahieren (wie in deinem RadarReader)
# Wir nehmen Antenne 0 und den gewählten Range-Bin
# FFT über die Samples pro Chirp
range_fft = np.fft.fft(data[:, 0, :, :], axis=-1)
bin_values = range_fft[:, :, 1] # Alle Chirps des Bins
    
    # Mittelwert über Chirps und Phase berechnen
    #print(bin_values.shape

complex_mean_all = np.mean(bin_values, axis=1)
phase_signal_all = np.angle(complex_mean_all)

unwrapped_phase_all = np.unwrap(phase_signal_all)

RADAR_FREQ_GHZ = 60.75  # e.g. 60 or 77

c = 3e8                                     # speed of light m/s
wavelength_m = c / (RADAR_FREQ_GHZ * 1e9)  # in meters
wavelength_mm = wavelength_m * 1000         # in mm
displacement_mm_all = (wavelength_mm * unwrapped_phase_all) / (4 * np.pi)
_, wave_all = extract_middle_pulse_waveform(displacement_mm_all, fps)
wave_all = (wave_all - np.min(wave_all)) / (np.max(wave_all) - np.min(wave_all))
print(wave_all.shape)



for i, filename in enumerate(files):
    ax = axes_flat[i]

    # 1. Laden
    file_path = os.path.join(input_folder, filename)
    data = np.load(file_path)

    num_frames, num_ant, num_chirps, num_samples = data.shape
    fps = num_frames/time 
    plot_chirps(ax, file_path, fps, wave_all)

    if i % 3 != 0: ax.set_ylabel("") 
    if i < 15: ax.set_xlabel("")


plt.tight_layout(pad=3.0) # Erhöht den Abstand zwischen den Plots
plt.subplots_adjust(top=0.95) # Platz für eine Hauptüberschrift
fig.suptitle(f"{input_folder}", fontsize=20)

plt.show()

### SNR based on Chirps used to average over


In [ ]:
input_folder = 'radar_session_2026-03-06_17-39_seb'
input_folder = 'radar_session_2026-03-06_16-57_nima'
input_folder = 'radar_session_2026-03-06_15-18_seb'
#input_folder = 'radar_invivo_fps_sweep_2026-05-20_12-33'
input_folders = [
    #'radar_session_2026-03-06_17-39_seb',
    #'radar_session_2026-03-06_15-18_seb',
    #'radar_session_2026-03-06_16-57_nima',
    #'radar_session_2026-03-06_14-38_nima',
    #'radar_session_2026-03-06_17-27_seb',
    #'radar_session_2026-03-06_17-15_ben',
    path_devkit_folder
]


# Parameter (bleiben für alle gleich zum Vergleich)

target_bin_index = 1
time = 10

# Alle .npy Dateien finden

plt.figure(figsize=(12, 7))
all_snr_norm = []
max_chirps = 512  # oder automatisch bestimmen

for input_folder in input_folders:

    files = sorted([f for f in os.listdir(input_folder) if f.endswith('.npy')])

    for filename in files:
        # 1. Laden
        file_path = os.path.join(input_folder, filename)
        data = np.load(file_path)

        # Erwartet (Frames, Antennas, Chirps, Samples)
        num_frames, num_ant, num_chirps, num_samples = data.shape
        fps = num_frames/time 
        

        # Phase extrahieren (wie in deinem RadarReader)
        # Wir nehmen Antenne 0 und den gewählten Range-Bin
        # FFT über die Samples pro Chirp
        range_fft = np.fft.fft(data[:, 0, :, :], axis=-1)
        bin_values = range_fft[:, :, target_bin_index] # Alle Chirps des Bins
        
        snr = []

        # Mittelwert über Chirps und Phase berechnen
        # print(bin_values.shape)
        # print(bin_values[:, :32].shape)
        for i in range(1, min(max_chirps+1,num_chirps+ 1)):
            complex_mean_1 = np.mean(bin_values[:, :i], axis=1)
            phase_signal_1 = np.angle(complex_mean_1)

            #Unwrapping um 2pi-Sprünge zu entfernen
            unwrapped_phase_1 = np.unwrap(phase_signal_1)

            RADAR_FREQ_GHZ = 60.75  # e.g. 60 or 77

            c = 3e8                                     # speed of light m/s
            wavelength_m = c / (RADAR_FREQ_GHZ * 1e9)  # in meters
            wavelength_mm = wavelength_m * 1000         # in mm

            # Convert unwrapped phase (radians) → displacement (mm)
            displacement_mm_1 = (wavelength_mm * unwrapped_phase_1) / (4 * np.pi)


            signal = bandpass_filter(displacement_mm_1, fps, lowcut=0.5, highcut=6, order=3)
            noise = bandpass_filter(displacement_mm_1, fps, lowcut=6, highcut=12, order=3)

            p_signal = np.mean(signal ** 2)
            p_noise  = np.mean(noise ** 2)

            snr.append(10 * np.log10(p_signal / p_noise))

        #label_name = filename.split('_')[1] if '_' in filename else filename

        snr = np.array(snr)

        snr = (snr - np.min(snr)) / (np.max(snr) - np.min(snr) + 1e-12)

        padded = np.full(max_chirps, np.nan)
        padded[:num_chirps] = snr

        all_snr_norm.append(padded)

In [ ]:
all_snr_norm = np.array(all_snr_norm)
# 3. Styling NACHDEM alle Linien gezeichnet wurden
chirp_axis = np.arange(1, max_chirps + 1)

mean_snr = np.nanmean(all_snr_norm, axis=0)
lower = np.nanpercentile(all_snr_norm, 25, axis=0)
upper = np.nanpercentile(all_snr_norm, 75, axis=0)

# Anzahl Kurven pro Chirp, wichtig zur Kontrolle
n_available = np.sum(~np.isnan(all_snr_norm), axis=0)
plt.figure(figsize=(9, 4))

plt.plot(
    chirp_axis,
    mean_snr,
    linewidth=2.5,
    label="Mean normalized SNR gain"
)

plt.fill_between(
    chirp_axis,
    lower,
    upper,
    alpha=0.25,
    label="25–75 percentile"
)

plt.axvline(
    x=32,
    linestyle="--",
    linewidth=1.5,
    label="Selected configuration: 32 chirps"
)

plt.xlabel("Number of chirps used for averaging")
plt.ylabel("Normalized SNR gain")
plt.ylim(0, 1.05)
plt.xlim(0, 512)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()


std_snr = np.nanstd(all_snr_norm, axis=0)

plt.figure(figsize=(9, 5))

plt.plot(
    chirp_axis,
    mean_snr,
    linewidth=2.5,
    label="Mean normalized SNR gain"
)

plt.fill_between(
    chirp_axis,
    mean_snr - std_snr,
    mean_snr + std_snr,
    alpha=0.25,
    label="±1 std"
)

plt.axvline(
    x=32,
    linestyle="--",
    linewidth=1.5,
    label="Selected configuration: 32 chirps"
)

plt.xlabel("Number of chirps used for averaging")
plt.ylabel("Normalized SNR gain")
plt.ylim(0, 1.05)
plt.xlim(0,256)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
input_folder = 'radar_session_2026-03-06_17-39_seb'
input_folder = 'radar_invivo_fps_sweep_2026-05-20_12-33'
input_folder = path_devkit_folder



# Parameter (bleiben für alle gleich zum Vergleich)

target_bin_index = 1
time = 10

# Alle .npy Dateien finden
files = sorted([f for f in os.listdir(input_folder) if f.endswith('.npy')])


plt.figure(figsize=(4, 4))

for filename in files:
    # 1. Laden
    file_path = os.path.join(input_folder, filename)
    data = np.load(file_path)

    # Erwartet (Frames, Antennas, Chirps, Samples)
    num_frames, num_ant, num_chirps, num_samples = data.shape
    fps = num_frames/time 
    

    # Phase extrahieren (wie in deinem RadarReader)
    # Wir nehmen Antenne 0 und den gewählten Range-Bin
    # FFT über die Samples pro Chirp
    range_fft = np.fft.fft(data[:, 0, :, :], axis=-1)
    bin_values = range_fft[:, :, target_bin_index] # Alle Chirps des Bins
    
    snr = []

    # Mittelwert über Chirps und Phase berechnen
    # print(bin_values.shape)
    # print(bin_values[:, :32].shape)
    for i in range(1, num_chirps + 1):
        complex_mean_1 = np.mean(bin_values[:, :i], axis=1)
        phase_signal_1 = np.angle(complex_mean_1)

        #Unwrapping um 2pi-Sprünge zu entfernen
        unwrapped_phase_1 = np.unwrap(phase_signal_1)

        RADAR_FREQ_GHZ = 60.75  # e.g. 60 or 77

        c = 3e8                                     # speed of light m/s
        wavelength_m = c / (RADAR_FREQ_GHZ * 1e9)  # in meters
        wavelength_mm = wavelength_m * 1000         # in mm

        # Convert unwrapped phase (radians) → displacement (mm)
        displacement_mm_1 = (wavelength_mm * unwrapped_phase_1) / (4 * np.pi)


        signal = bandpass_filter(displacement_mm_1, fps, lowcut=0.5, highcut=6, order=3)
        noise = bandpass_filter(displacement_mm_1, fps, lowcut=6, highcut=12, order=3)

        p_signal = np.mean(signal ** 2)
        p_noise  = np.mean(noise ** 2)

        snr.append(10 * np.log10(p_signal / p_noise))

    #label_name = filename.split('_')[1] if '_' in filename else filename
    label_name = filename
    plt.plot(range(1, num_chirps + 1), snr, label=f"{label_name})", alpha=0.8)

# 3. Styling NACHDEM alle Linien gezeichnet wurden

plt.title(f"SNR vs Chirps used to average",  fontsize=12, fontweight='bold')
plt.xlabel("Chirps")
plt.ylim(19,21)
plt.ylabel("SNR (dB)")
plt.grid(True, linestyle='--', alpha=0.6)
#plt.xscale('log')
plt.show()

In [ ]:
input_folder = 'radar_session_2026-03-06_17-39_seb'
input_folder = 'radar_session_2026-03-06_15-18_seb'
input_folder = 'radar_session_2026-03-06_16-57_nima'
input_folder = 'radar_session_2026-03-06_14-38_nima'
input_folder = 'radar_session_2026-03-06_17-27_seb'
input_folder = 'radar_session_2026-03-06_17-15_ben'
#input_folder = 'radar_invivo_fps_sweep_2026-05-20_12-33'
input_folder = path_devkit_folder


# Parameter (bleiben für alle gleich zum Vergleich)

target_bin_index = 1
time = 10

# Alle .npy Dateien finden
files = sorted([f for f in os.listdir(input_folder) if f.endswith('.npy')])

plt.figure(figsize=(12, 7))

for filename in files:
    # 1. Laden
    file_path = os.path.join(input_folder, filename)
    data = np.load(file_path)

    # Erwartet (Frames, Antennas, Chirps, Samples)
    num_frames, num_ant, num_chirps, num_samples = data.shape
    fps = num_frames/time 
    

    # Phase extrahieren (wie in deinem RadarReader)
    # Wir nehmen Antenne 0 und den gewählten Range-Bin
    # FFT über die Samples pro Chirp
    range_fft = np.fft.fft(data[:, 0, :, :], axis=-1)
    bin_values = range_fft[:, :, target_bin_index] # Alle Chirps des Bins
    
    snr = []

    # Mittelwert über Chirps und Phase berechnen
    # print(bin_values.shape)
    # print(bin_values[:, :32].shape)
    for i in range(1, num_chirps + 1):
        complex_mean_1 = np.mean(bin_values[:, :i], axis=1)
        phase_signal_1 = np.angle(complex_mean_1)

        #Unwrapping um 2pi-Sprünge zu entfernen
        unwrapped_phase_1 = np.unwrap(phase_signal_1)

        RADAR_FREQ_GHZ = 60.75  # e.g. 60 or 77

        c = 3e8                                     # speed of light m/s
        wavelength_m = c / (RADAR_FREQ_GHZ * 1e9)  # in meters
        wavelength_mm = wavelength_m * 1000         # in mm

        # Convert unwrapped phase (radians) → displacement (mm)
        displacement_mm_1 = (wavelength_mm * unwrapped_phase_1) / (4 * np.pi)


        signal = bandpass_filter(displacement_mm_1, fps, lowcut=0.5, highcut=6, order=3)
        noise = bandpass_filter(displacement_mm_1, fps, lowcut=6, highcut=12, order=3)

        p_signal = np.mean(signal ** 2)
        p_noise  = np.mean(noise ** 2)

        snr.append(10 * np.log10(p_signal / p_noise))

    #label_name = filename.split('_')[1] if '_' in filename else filename
    label_name = filename
    plt.plot(range(1, num_chirps + 1), snr, label=f"{label_name})", alpha=0.8)

# 3. Styling NACHDEM alle Linien gezeichnet wurden
plt.xlabel("Chirps", fontsize=12)


plt.ylabel("SNR (dB)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
#plt.xscale('log')

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small') # Legende außen platzieren
plt.tight_layout()
plt.show()

In [ ]:
input_folder = 'radar_session_2026-03-06_17-39_seb'
input_folder = path_devkit_folder
# Parameter (bleiben für alle gleich zum Vergleich)

target_bin_index = 1
time = 10

# Alle .npy Dateien finden
files = sorted([f for f in os.listdir(input_folder) if f.endswith('.npy')])

#plt.figure(figsize=(12, 7))
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes_flat = axes.flatten()


for filename in files:
    # 1. Laden
    file_path = os.path.join(input_folder, filename)
    data = np.load(file_path)

    # Erwartet (Frames, Antennas, Chirps, Samples)
    num_frames, num_ant, num_chirps, num_samples = data.shape
    fps = num_frames/time 

    if fps == 200:
        ax = axes_flat[0]
        ax.set_title(f"200FPS", fontsize=10)
    if fps == 100:
        ax = axes_flat[1]
        ax.set_title(f"100FPS", fontsize=10)
    if fps == 50:
        ax = axes_flat[2]
        ax.set_title(f"50FPS", fontsize=10)
    if fps == 25:
        ax = axes_flat[3]
        ax.set_title(f"25FPS", fontsize=10)
    
    # Phase extrahieren (wie in deinem RadarReader)
    # Wir nehmen Antenne 0 und den gewählten Range-Bin
    # FFT über die Samples pro Chirp
    range_fft = np.fft.fft(data[:, 0, :, :], axis=-1)
    bin_values = range_fft[:, :, target_bin_index] # Alle Chirps des Bins
    
    snr = []

    # print(bin_values.shape)
    # print(bin_values[:, :32].shape)
    for i in range(1, num_chirps + 1):
        complex_mean_1 = np.mean(bin_values[:, :i], axis=1)
        phase_signal_1 = np.angle(complex_mean_1)

        #Unwrapping um 2pi-Sprünge zu entfernen
        unwrapped_phase_1 = np.unwrap(phase_signal_1)

        RADAR_FREQ_GHZ = 60.75  # e.g. 60 or 77

        c = 3e8                                     # speed of light m/s
        wavelength_m = c / (RADAR_FREQ_GHZ * 1e9)   # in meters
        wavelength_mm = wavelength_m * 1000         # in mm

        # Convert unwrapped phase (radians) → displacement (mm)
        displacement_mm_1 = (wavelength_mm * unwrapped_phase_1) / (4 * np.pi)


        signal = bandpass_filter(displacement_mm_1, fps, lowcut=0.5, highcut=6, order=3)
        noise = bandpass_filter(displacement_mm_1, fps, lowcut=6, highcut=12, order=3)

        p_signal = np.mean(signal ** 2)
        p_noise  = np.mean(noise ** 2)

        snr.append(10 * np.log10(p_signal / p_noise))

    #label_name = filename.split('_')[1] if '_' in filename else filename
    label_name = filename
    ax.plot(range(1, num_chirps + 1), snr, label=f"{label_name})", alpha=0.8)
    
    ax.set_ylabel("SNR (dB)", fontsize=12)
    ax.set_xlabel("Chirps", fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(bbox_to_anchor=(0.5, -0.2), loc='upper center', fontsize='small') # Legende außen platzieren
    ax.set_xlim(1, 64)
    ax.set_ylim(19, 21)

plt.tight_layout()
plt.subplots_adjust(top=0.9)
fig.suptitle(f"{input_folder}", fontsize=20)


## Below is legacy stuff, looked at some cosine similarity but nothing was found really

In [ ]:
def extract_middle_pulse_waveform_filterd(signal, fps, num_points=100):
    # 1. Filtern
    filtered = bandpass_filter(signal, fs=fps)
    #filtered = signal
    # 2. Peaks finden (mit Prominence für Stabilität)
    min_peak_distance = max(1, int(0.4 * fps))
    prominence = max(1e-9, 0.2 * np.std(filtered))
    peaks, _ = find_peaks(filtered, distance=min_peak_distance, prominence=prominence)
    
    if len(peaks) < 3: return None

    # 3. Segmente extrahieren und auf 100 Punkte resamplen
    beats = []
    for i in range(len(peaks) - 1):
        segment = filtered[peaks[i]:peaks[i+1]]
        if len(segment) < 4: continue
        
        x_old = np.linspace(0, 1, len(segment))
        x_new = np.linspace(0, 1, num_points)
        segment_resampled = np.interp(x_new, x_old, segment)
        beats.append(segment_resampled)
    
    if len(beats) < 2: return None
    
    beats = np.vstack(beats)
    mean_waveform = np.mean(beats, axis=0)
    
    # 4. Invertierung prüfen (Soll nach oben zeigen)
    k = max(3, int(0.10 * num_points))
    slope = np.polyfit(np.arange(k), mean_waveform[:k], 1)[0]
    if slope < 0:
        beats = -beats

    correlations = np.array([
        np.mean([np.corrcoef(beats[i], beats[j])[0, 1] 
                 for j in range(len(beats)) if i != j])
        for i in range(len(beats))
    ])
    median_beat = beats[np.argmax(correlations)]

    return np.linspace(0, 100, num_points), median_beat

def get_wave(range_fft, fps, chirps, range_bin=1):
    # Daten laden (shape: frames, antennas, chirps, samples)

    
    bin_values = range_fft[:, :, range_bin] # Alle Chirps des Bins
    
    complex_mean_1 = np.mean(bin_values[:, :chirps], axis=1)
    phase_signal_1 = np.angle(complex_mean_1)
    
    # WICHTIG: Unwrapping um 2pi-Sprünge zu entfernen
    unwrapped_phase_1 = np.unwrap(phase_signal_1)


    RADAR_FREQ_GHZ = 60.75  # e.g. 60 or 77

    c = 3e8                                     # speed of light m/s
    wavelength_m = c / (RADAR_FREQ_GHZ * 1e9)  # in meters
    wavelength_mm = wavelength_m * 1000         # in mm

    # Convert unwrapped phase (radians) → displacement (mm)
    displacement_mm_1 = (wavelength_mm * unwrapped_phase_1) / (4 * np.pi)
        
    _, wave_1 = extract_middle_pulse_waveform_filterd(displacement_mm_1, fps)

    wave_1 = (wave_1 - np.min(wave_1)) / (np.max(wave_1) - np.min(wave_1))

    return wave_1


def get_ref(file_path, fps, range_bin=1):
    # Daten laden (shape: frames, antennas, chirps, samples)
    data = np.load(file_path)
    
    # Phase extrahieren (wie in deinem RadarReader)
    # Wir nehmen Antenne 0 und den gewählten Range-Bin
    # FFT über die Samples pro Chirp
    range_fft = np.fft.fft(data[:, 0, :, :], axis=-1)
    bin_values = range_fft[:, :, range_bin] # Alle Chirps des Bins
    
    # Mittelwert über Chirps und Phase berechnen
    #print(bin_values.shape)
   # print(bin_values[:, :32].shape)
    complex_mean_all = np.mean(bin_values, axis=1)
    phase_signal_all = np.angle(complex_mean_all)
    
    # WICHTIG: Unwrapping um 2pi-Sprünge zu entfernen
    unwrapped_phase_all = np.unwrap(phase_signal_all)

    #filtered_ref_1 = bandpass_filter(unwrapped_phase_all, fps)

    RADAR_FREQ_GHZ = 60.75  # e.g. 60 or 77

    c = 3e8                                     # speed of light m/s
    wavelength_m = c / (RADAR_FREQ_GHZ * 1e9)   # in meters
    wavelength_mm = wavelength_m * 1000         # in mm

    # Convert unwrapped phase (radians) → displacement (mm)
    displacement_mm_all = (wavelength_mm * unwrapped_phase_all) / (4 * np.pi)
        
    #print(displacement_mm_1.shape, '  ',displacement_mm_32.shape, '  ',displacement_mm_all.shape,)

    # Durchschnittlichen Beat berechnen
    _, wave_1 = extract_middle_pulse_waveform_filterd(displacement_mm_all, fps)

        
    wave_1 = (wave_1 - np.min(wave_1)) / (np.max(wave_1) - np.min(wave_1))

    return wave_1

def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0
    return np.dot(a, b) / denom


In [ ]:
input_folder = 'radar_session_2026-03-06_16-57_nima'
input_folder = path_devkit_folder
time = 10

max_chirps = 512

files = sorted([f for f in os.listdir(input_folder) if f.endswith('.npy')])

ref1 = get_ref(os.path.join(input_folder,'01_200fps_35chrep_128ch_32sa.npy'), 200)
ref2 = get_ref(os.path.join(input_folder,'09_200fps_35chrep_128ch_32sa.npy'), 200)
ref3 = get_ref(os.path.join(input_folder,'18_200fps_35chrep_128ch_32sa.npy'), 200)

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes_flat = axes.flatten()


for i, filename in enumerate(files):

    file_path = os.path.join(input_folder, filename)
    data = np.load(file_path)
    num_frames, num_ant, num_chirps, num_samples = data.shape
    fps = int(num_frames/time)

    if fps == 200:
        ax = axes_flat[0]
        ax.set_title("200 FPS", fontsize=10)
    if fps == 100:
        ax = axes_flat[1]
        ax.set_title("100 FPS", fontsize=10)
    if fps == 50:
        ax = axes_flat[2]
        ax.set_title("50 FPS", fontsize=10)
    if fps == 25:
        ax = axes_flat[3]
        ax.set_title("25 FPS", fontsize=10)

    output = []
    range_fft = np.fft.fft(data[:, 0, :, :], axis=-1)

    for chirpcount in range(1, num_chirps+1):
       
        wave = get_wave(range_fft,fps,chirpcount)

        cos1 = cosine_similarity(ref1, wave)
        cos2 = cosine_similarity(ref2, wave)
        cos3 = cosine_similarity(ref3, wave)
        output.append((cos1+cos2+cos3)/3.0)

        #if chirpcount == max_chirps:
        #    break




    label_name = filename
    ax.plot(range(1, np.min([max_chirps + 1, num_chirps+1])), output, label=f"{label_name}", alpha=0.8)
    
    ax.set_xlabel("Chirps", fontsize=12)
    ax.set_ylabel("Cosine Similarity", fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(bbox_to_anchor=(0.5, -0.2), loc='upper center', fontsize='small') # Legende außen platzieren
    #ax.set_xlim(1, 64)
    ax.set_ylim(0.980, 1)
    
    
plt.tight_layout(pad=3.0) # Erhöht den Abstand zwischen den Plots
plt.subplots_adjust(top=0.95) # Platz für eine Hauptüberschrift
plt.show()

In [ ]:
def extract_average_pulse_waveform_filterd(signal, fps, num_points=100):
    # 1. Filtern
    filtered = bandpass_filter(signal, fs=fps)
    
    #filtered = signal
    # 2. Peaks finden (mit Prominence für Stabilität)
    min_peak_distance = max(1, int(0.4 * fps))
    prominence = max(1e-9, 0.2 * np.std(filtered))
    peaks, _ = find_peaks(filtered, distance=min_peak_distance, prominence=prominence)
    
    if len(peaks) < 3: return None

    # 3. Segmente extrahieren und auf 100 Punkte resamplen
    beats = []
    for i in range(len(peaks) - 1):
        segment = filtered[peaks[i]:peaks[i+1]]
        if len(segment) < 4: continue
        
        x_old = np.linspace(0, 1, len(segment))
        x_new = np.linspace(0, 1, num_points)
        segment_resampled = np.interp(x_new, x_old, segment)
        beats.append(segment_resampled)
    
    if len(beats) < 2: return None
    
    beats = np.vstack(beats)
    mean_waveform = np.mean(beats, axis=0)
    
    # 4. Invertierung prüfen (Soll nach oben zeigen)
    k = max(3, int(0.10 * num_points))
    slope = np.polyfit(np.arange(k), mean_waveform[:k], 1)[0]
    if slope < 0:
        beats = -beats
        mean_waveform = np.mean(beats, axis=0)

    return np.linspace(0, 100, num_points), mean_waveform

def get_wave_mean(range_fft, fps, chirps, range_bin=1):
    # Daten laden (shape: frames, antennas, chirps, samples)

    
    bin_values = range_fft[:, :, range_bin] # Alle Chirps des Bins
    
    complex_mean_1 = np.mean(bin_values[:, :chirps], axis=1)
    phase_signal_1 = np.angle(complex_mean_1)
    
    # WICHTIG: Unwrapping um 2pi-Sprünge zu entfernen
    unwrapped_phase_1 = np.unwrap(phase_signal_1)


    RADAR_FREQ_GHZ = 60.75  # e.g. 60 or 77

    c = 3e8                                     # speed of light m/s
    wavelength_m = c / (RADAR_FREQ_GHZ * 1e9)  # in meters
    wavelength_mm = wavelength_m * 1000         # in mm

    # Convert unwrapped phase (radians) → displacement (mm)
    displacement_mm_1 = (wavelength_mm * unwrapped_phase_1) / (4 * np.pi)
        
    _, wave_1 = extract_average_pulse_waveform_filterd(displacement_mm_1, fps)

    wave_1 = (wave_1 - np.min(wave_1)) / (np.max(wave_1) - np.min(wave_1))

    return wave_1


def get_ref_mean(file_path, fps, range_bin=1):
    # Daten laden (shape: frames, antennas, chirps, samples)
    data = np.load(file_path)
    
    # Phase extrahieren (wie in deinem RadarReader)
    # Wir nehmen Antenne 0 und den gewählten Range-Bin
    # FFT über die Samples pro Chirp
    range_fft = np.fft.fft(data[:, 0, :, :], axis=-1)
    bin_values = range_fft[:, :, range_bin] # Alle Chirps des Bins
    
    # Mittelwert über Chirps und Phase berechnen
    #print(bin_values.shape)
   # print(bin_values[:, :32].shape)
    complex_mean_all = np.mean(bin_values, axis=1)
    phase_signal_all = np.angle(complex_mean_all)
    
    # WICHTIG: Unwrapping um 2pi-Sprünge zu entfernen
    unwrapped_phase_all = np.unwrap(phase_signal_all)

    #filtered_ref_1 = bandpass_filter(unwrapped_phase_all, fps)

    RADAR_FREQ_GHZ = 60.75  # e.g. 60 or 77

    c = 3e8                                     # speed of light m/s
    wavelength_m = c / (RADAR_FREQ_GHZ * 1e9)   # in meters
    wavelength_mm = wavelength_m * 1000         # in mm

    # Convert unwrapped phase (radians) → displacement (mm)
    displacement_mm_all = (wavelength_mm * unwrapped_phase_all) / (4 * np.pi)
        
    #print(displacement_mm_1.shape, '  ',displacement_mm_32.shape, '  ',displacement_mm_all.shape,)

    # Durchschnittlichen Beat berechnen
    _, wave_1 = extract_average_pulse_waveform_filterd(displacement_mm_all, fps)

        
    wave_1 = (wave_1 - np.min(wave_1)) / (np.max(wave_1) - np.min(wave_1))

    return wave_1

def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0
    return np.dot(a, b) / denom

In [ ]:
input_folder = 'radar_session_2026-03-06_16-57_nima'
input_folder = path_devkit_folder
time = 10

max_chirps = 512

files = sorted([f for f in os.listdir(input_folder) if f.endswith('.npy')])

ref1 = get_ref_mean(os.path.join(input_folder,'01_200fps_35chrep_128ch_32sa.npy'), 200)
ref2 = get_ref_mean(os.path.join(input_folder,'09_200fps_35chrep_128ch_32sa.npy'), 200)
ref3 = get_ref_mean(os.path.join(input_folder,'18_200fps_35chrep_128ch_32sa.npy'), 200)

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes_flat = axes.flatten()


for i, filename in enumerate(files):

    file_path = os.path.join(input_folder, filename)
    data = np.load(file_path)
    num_frames, num_ant, num_chirps, num_samples = data.shape
    fps = int(num_frames/time)

    if fps == 200:
        ax = axes_flat[0]
        ax.set_title("200 FPS", fontsize=10)
    if fps == 100:
        ax = axes_flat[1]
        ax.set_title("100 FPS", fontsize=10)
    if fps == 50:
        ax = axes_flat[2]
        ax.set_title("50 FPS", fontsize=10)
    if fps == 25:
        ax = axes_flat[3]
        ax.set_title("25 FPS", fontsize=10)

    output = []
    range_fft = np.fft.fft(data[:, 0, :, :], axis=-1)

    for chirpcount in range(1, num_chirps+1):
       
        wave = get_wave_mean(range_fft,fps,chirpcount)

        cos1 = cosine_similarity(ref1, wave)
        cos2 = cosine_similarity(ref2, wave)
        cos3 = cosine_similarity(ref3, wave)
        output.append((cos1+cos2+cos3)/3.0)

        #if chirpcount == max_chirps:
        #    break




    label_name = filename
    ax.plot(range(1, np.min([max_chirps + 1, num_chirps+1])), output, label=f"{label_name}", alpha=0.8)
    
    ax.set_xlabel("Chirps", fontsize=12)
    ax.set_ylabel("Cosine Similarity", fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(bbox_to_anchor=(0.5, -0.2), loc='upper center', fontsize='small') # Legende außen platzieren
    #ax.set_xlim(1, 64)
    ax.set_ylim(0.9925, 1)
    
    
plt.tight_layout(pad=3.0) # Erhöht den Abstand zwischen den Plots
plt.subplots_adjust(top=0.95) # Platz für eine Hauptüberschrift
plt.show()